# 02 - Accommodation Data Collection

## Objective

Collect accommodation data for the 50 initial destinations
defined in the destination master dataset.

This is the next data-collection stage of the Travel Agent project.

## Completed Before This Notebook

The following data pipelines are already completed:

- Destination Master
- Destination Locations
- Places Data
- Places Cleaning
- Places Feature Engineering
- Places Model Features
- Weather Data
- Weather Integration

Final integrated dataset:

`../data/cleaned/travel_features.csv`

## Current Approach

We will now build the accommodation pipeline:

Destination Master
        ↓
Accommodation Data Collection
        ↓
Raw Accommodation Dataset
        ↓
Data Validation
        ↓
Data Cleaning
        ↓
Accommodation Features
        ↓
Accommodation Price Features
        ↓
Final Accommodation Dataset

## Initial Scope

For development and testing, we are using the same
50 destinations already defined in the project.

Later, this exact pipeline will be expanded to cover
broader destinations across India.

## Important

This notebook is dedicated only to accommodation data.

We will complete and validate the accommodation approach
before starting the Flights approach.

## Expected Outputs

Raw:

`../data/raw/accommodation/`

Cleaned:

`../data/cleaned/accommodation_cleaned.csv`

Features:

`../data/cleaned/accommodation_features.csv`

In [12]:

import pandas as pd

import os

pd.set_option("display.max_columns", None)

pd.set_option("display.width", 1000)

In [13]:
# Load the destination master dataset that was created
destination_master = pd.read_csv(
    "../data/raw/destination_master.csv"
)

# Display the basic shape of the dataset.
print("Rows:", len(destination_master))
print("Columns:", destination_master.columns.tolist())

# Check the first few destinations to confirm the file loaded correctly.
display(destination_master.head())

Rows: 50
Columns: ['destination_id', 'destination', 'search_query']


,destination_id,destination,search_query
0,1,Goa,Goa
1,2,Munnar,Munnar
2,3,Manali,Manali
3,4,Jaipur,Jaipur
4,5,Udaipur,Udaipur


In [14]:
# Check that the destination master contains exactly 50 destinations.
print("Total destinations:", len(destination_master))

# Check whether every destination name is unique.
print("Unique destinations:", destination_master["destination"].nunique())

# Check for duplicate destination names.
duplicate_destinations = destination_master[
    destination_master["destination"].duplicated(keep=False)
]

print("\nDuplicate destinations:")
display(duplicate_destinations)

# Check for missing values in the destination master.
print("\nMissing values:")
print(destination_master.isnull().sum())

Total destinations: 50
Unique destinations: 50

Duplicate destinations:


,destination_id,destination,search_query



Missing values:
destination_id    0
destination       0
search_query      0
dtype: int64


In [15]:
# Define the folder where raw accommodation API/data records will be stored.
accommodation_raw_dir = "../data/raw/accommodation"

# Create the folder if it does not already exist.
# exist_ok=True prevents an error if the folder is already present.
os.makedirs(accommodation_raw_dir, exist_ok=True)

print("Accommodation raw-data folder ready:")
print(accommodation_raw_dir)

Accommodation raw-data folder ready:
../data/raw/accommodation


# Accommodation Collection Strategy

## Objective

Collect accommodation information for each of the 50 initial
destinations.

The accommodation data must eventually support two requirements
of our Travel Agent:

1. Showing accommodation options to the user.
2. Estimating accommodation cost for the complete trip budget.

## Information We Need

For each accommodation record, we will try to collect:

- Destination
- Hotel/accommodation name
- Hotel ID, if provided
- Rating, if available
- Price
- Currency
- Accommodation type, if available
- Location
- Other useful hotel information

## Collection Flow

Destination
    ↓
Find accommodation options
    ↓
Collect hotel records
    ↓
Store raw API response/data
    ↓
Validate records
    ↓
Clean accommodation data
    ↓
Extract price information
    ↓
Create destination-level accommodation features

## Important

Accommodation prices are dynamic and can depend on:

- Check-in date
- Check-out date
- Number of guests
- Room availability

Therefore, we will keep the raw accommodation records separate
from the destination-level features.

This allows us to use the detailed records later when the final
Travel Agent receives actual user travel dates.

## Initial Scope

We are collecting data for the same 50 destinations used throughout
the current development phase.

The code will be designed so the same pipeline can later be expanded
to broader destinations across India.

## Expected Outputs

Raw accommodation data:

`../data/raw/accommodation/accommodation_raw.csv`

Cleaned accommodation data:

`../data/cleaned/accommodation_cleaned.csv`

Destination-level accommodation features:

`../data/cleaned/accommodation_features.csv`

## Completion Condition

We will consider the Accommodation approach complete only after:

- Data collection is completed
- Destinations are validated
- Duplicate records are checked
- Missing values are analyzed
- Prices are checked
- Destination-level features are created
- Final accommodation data is saved

After that, we will move to the Flights approach.

In [16]:
# Load environment variables from the project's .env file.
from dotenv import load_dotenv

load_dotenv("../.env")

# Read the Amadeus credentials without printing the actual secret values.
# We only check whether they exist.
AMADEUS_CLIENT_ID = os.getenv("AMADEUS_CLIENT_ID")
AMADEUS_CLIENT_SECRET = os.getenv("AMADEUS_CLIENT_SECRET")

print("Amadeus Client ID available:", bool(AMADEUS_CLIENT_ID))
print("Amadeus Client Secret available:", bool(AMADEUS_CLIENT_SECRET))

Amadeus Client ID available: False
Amadeus Client Secret available: False


In [17]:
# Load the credentials stored in the project's .env file.
# The actual API key and secret are never printed.
from dotenv import load_dotenv

load_dotenv("../.env")

# Read Hotelbeds credentials from environment variables.
HOTELBEDS_API_KEY = os.getenv("HOTELBEDS_API_KEY")
HOTELBEDS_SECRET = os.getenv("HOTELBEDS_SECRET")

# Verify only whether the credentials exist.
print("Hotelbeds API Key available:", bool(HOTELBEDS_API_KEY))
print("Hotelbeds Secret available:", bool(HOTELBEDS_SECRET))

Hotelbeds API Key available: True
Hotelbeds Secret available: True


In [18]:
# Import libraries required to create the Hotelbeds authentication signature.
import hashlib
import time

# Create the current Unix timestamp.
# Hotelbeds uses the current timestamp as part of the X-Signature.
timestamp = str(int(time.time()))

# Combine the API key, secret, and timestamp.
signature_string = HOTELBEDS_API_KEY + HOTELBEDS_SECRET + timestamp

# Generate the SHA-256 hash required by Hotelbeds.
hotelbeds_signature = hashlib.sha256(
    signature_string.encode("utf-8")
).hexdigest()

# Show only safe information for verification.
# We never print the API key, secret, or complete signature.
print("Hotelbeds timestamp generated:", timestamp)
print("Hotelbeds signature generated:", bool(hotelbeds_signature))

Hotelbeds timestamp generated: 1787836789
Hotelbeds signature generated: True


In [19]:
# Import requests so we can send an HTTP request to Hotelbeds.
import requests

# Hotelbeds provides a dedicated status endpoint for testing
# whether our API credentials and authentication are working.
status_url = "https://api.test.hotelbeds.com/hotel-api/1.0/status"

# Prepare the required authentication headers.
# The API key identifies our account.
# The X-Signature authenticates the request using our secret.
headers = {
    "Accept": "application/json",
    "Api-key": HOTELBEDS_API_KEY,
    "X-Signature": hotelbeds_signature
}

# Send a simple GET request to the Hotelbeds test status endpoint.
response = requests.get(
    status_url,
    headers=headers,
    timeout=30
)

# Print only safe response information.
print("Status code:", response.status_code)
print("Response:")
print(response.text)

Status code: 200
Response:
{"auditData":{"timestamp":"2026-08-27 13:19:51.075"},"status":"OK"}


# Hotel Search — Single Destination Test

Authentication is working successfully.

Before collecting accommodation data for all 50 destinations,
we will test the hotel availability API with one destination.

## Test Objective

Verify that we can:

1. Send a hotel search request.
2. Receive hotel records.
3. Read hotel names and hotel information.
4. Read room/availability information.
5. Read price information.
6. Understand the response structure.

## Test Destination

We will use:

**Goa**

Goa is already present in our destination master and has
location information from the previous data-collection stage.

## Important

We are intentionally testing only ONE destination first.

We will NOT collect all 50 destinations until the response
structure has been inspected and validated.

## Flow

Goa
 ↓
Hotelbeds Hotel Search API
 ↓
Hotel response
 ↓
Inspect response structure
 ↓
Identify useful fields
 ↓
Design raw accommodation dataset

In [20]:
# Load the destination location data created in the previous notebook.
# This contains the validated latitude and longitude for our 50 destinations.
destination_locations = pd.read_csv(
    "../data/raw/places/destination_locations.csv"
)

# Check the dataset before using it for the accommodation API.
print("Rows:", len(destination_locations))
print("Columns:", destination_locations.columns.tolist())

# Display the first few records.
display(destination_locations.head())

Rows: 50
Columns: ['destination_id', 'destination', 'search_query', 'resolved_name', 'country', 'country_code', 'state', 'state_code', 'latitude', 'longitude', 'result_type', 'formatted_address', 'confidence', 'match_type', 'place_id']


,destination_id,destination,search_query,resolved_name,country,country_code,state,state_code,latitude,longitude,result_type,formatted_address,confidence,match_type,place_id
0,1,Goa,Goa,Goa,India,in,Goa,GA,15.300454,74.085513,state,"Goa, India",1.0,full_match,51aa17320d798552405999e26025d5992e40f00101f901...
1,2,Munnar,Munnar,Munnar,India,in,Kerala,KL,10.086996,77.060091,city,"Munnar, KL, India",1.0,full_match,5120d1048ad843534059adc502ba8a2c2440f00103f901...
2,3,Manali,Manali,Manali,India,in,Himachal Pradesh,HP,32.245461,77.187293,city,"Manali, HP, India",1.0,full_match,511af2199afc4b53405999396e426b1f4040f00103f901...
3,4,Jaipur,Jaipur,Jaipur,India,in,Rajasthan,RJ,26.915458,75.818982,city,"Jaipur, RJ, India",1.0,full_match,51706138326af4524059e9dfe46d5bea3a40f00103f901...
4,5,Udaipur,Udaipur,Udaipur,India,in,Rajasthan,RJ,24.578721,73.686257,city,"Udaipur, RJ, India",1.0,full_match,517649e6a2eb6b5240592882380f27943840f00103f901...


In [21]:
# Select Goa from our validated destination-location dataset.
# We use the existing coordinates instead of manually entering them.
goa_location = destination_locations[
    destination_locations["destination"].str.lower() == "goa"
].iloc[0]

# Extract Goa's latitude and longitude.
goa_latitude = goa_location["latitude"]
goa_longitude = goa_location["longitude"]

print("Destination:", goa_location["destination"])
print("Latitude:", goa_latitude)
print("Longitude:", goa_longitude)

Destination: Goa
Latitude: 15.3004543
Longitude: 74.0855134


In [22]:
# Define a simple one-night test stay.
# We use a future date so that we are requesting actual
# accommodation availability rather than historical data.

check_in = "2026-09-15"
check_out = "2026-09-16"

# Prepare the Hotelbeds availability request.
# Coordinates come directly from our validated location dataset.
hotel_request = {
    "stay": {
        "checkIn": check_in,
        "checkOut": check_out
    },
    "occupancies": [
        {
            "rooms": 1,
            "adults": 1,
            "children": 0
        }
    ],
    "geolocation": {
        "latitude": float(goa_latitude),
        "longitude": float(goa_longitude),
        "radius": 20,
        "unit": "km"
    }
}

print("Destination: Goa")
print("Check-in:", check_in)
print("Check-out:", check_out)
print("Guests: 1 adult")
print("Rooms: 1")
print("Search radius: 20 km")

Destination: Goa
Check-in: 2026-09-15
Check-out: 2026-09-16
Guests: 1 adult
Rooms: 1
Search radius: 20 km


In [38]:
# Hotelbeds hotel availability endpoint.
availability_url = (
    "https://api.test.hotelbeds.com/hotel-api/1.0/hotels"
)

# Send the hotel search request for Goa.
# The JSON body contains our stay dates, occupancy, and geographic search area.
availability_response = requests.post(
    availability_url,
    headers={
        "Accept": "application/json",
        "Content-Type": "application/json",
        "Api-key": HOTELBEDS_API_KEY,
        "X-Signature": hotelbeds_signature
    },
    json=hotel_request,
    timeout=60
)

# Display only the HTTP status first.
# We will inspect the actual response structure separately.
print("Status code:", availability_response.status_code)

# Display the response text so we can see what Hotelbeds returned.
print("\nResponse:")
print(availability_response.text[:5000])

Status code: 403

Response:
{
    "error": "Quota exceeded"
}


# Accommodation Raw Dataset Schema

The Hotelbeds response contains nested hotel → room → rate information.

We will flatten this information so that each row represents
one available hotel rate for a specific room.

## Main Fields

### Destination information
- destination
- destination_code
- zone_name

### Hotel information
- hotel_code
- hotel_name
- hotel_category
- hotel_category_name
- hotel_latitude
- hotel_longitude

### Room information
- room_code
- room_name

### Rate information
- rate_key
- rate_class
- rate_type
- price
- allotment
- payment_type
- board_code
- board_name
- rooms
- adults
- children

### Offer information
- offer_name
- offer_amount

### Cancellation information
- cancellation_amount
- cancellation_from

### Search information
- check_in
- check_out

## Important

We will preserve the `rate_key` because it identifies the
specific rate returned by Hotelbeds.

The raw API response will also be preserved separately so that
we do not lose information that may become useful later.

In [39]:
def parse_hotelbeds_response(response_json, destination, check_in, check_out):
    """
    Convert the nested Hotelbeds hotel response into a flat DataFrame.

    Each output row represents one hotel + room + rate combination.
    This makes the data easier to analyze, clean, and use later
    for accommodation price calculations.
    """

    records = []

    # Get the hotel list from the API response.
    hotels = response_json.get("hotels", {}).get("hotels", [])

    # Loop through every hotel returned by Hotelbeds.
    for hotel in hotels:

        # Basic hotel information.
        hotel_code = hotel.get("code")
        hotel_name = hotel.get("name")
        category_code = hotel.get("categoryCode")
        category_name = hotel.get("categoryName")
        destination_code = hotel.get("destinationCode")
        destination_name = hotel.get("destinationName")
        zone_code = hotel.get("zoneCode")
        zone_name = hotel.get("zoneName")

        hotel_latitude = hotel.get("latitude")
        hotel_longitude = hotel.get("longitude")

        # A hotel can contain multiple room types.
        for room in hotel.get("rooms", []):

            room_code = room.get("code")
            room_name = room.get("name")

            # A room can contain multiple rates.
            for rate in room.get("rates", []):

                # Basic rate information.
                record = {
                    "destination": destination,
                    "check_in": check_in,
                    "check_out": check_out,

                    "hotel_code": hotel_code,
                    "hotel_name": hotel_name,
                    "hotel_category": category_code,
                    "hotel_category_name": category_name,

                    "destination_code": destination_code,
                    "destination_name": destination_name,
                    "zone_code": zone_code,
                    "zone_name": zone_name,

                    "hotel_latitude": hotel_latitude,
                    "hotel_longitude": hotel_longitude,

                    "room_code": room_code,
                    "room_name": room_name,

                    "rate_key": rate.get("rateKey"),
                    "rate_class": rate.get("rateClass"),
                    "rate_type": rate.get("rateType"),

                    "price": rate.get("net"),
                    "allotment": rate.get("allotment"),

                    "payment_type": rate.get("paymentType"),
                    "board_code": rate.get("boardCode"),
                    "board_name": rate.get("boardName"),

                    "rooms": rate.get("rooms"),
                    "adults": rate.get("adults"),
                    "children": rate.get("children")
                }

                # Extract the first offer if one is available.
                offers = rate.get("offers", [])

                if offers:
                    record["offer_name"] = offers[0].get("name")
                    record["offer_amount"] = offers[0].get("amount")
                else:
                    record["offer_name"] = None
                    record["offer_amount"] = None

                # Extract the first cancellation policy if available.
                cancellation_policies = rate.get(
                    "cancellationPolicies", []
                )

                if cancellation_policies:
                    record["cancellation_amount"] = (
                        cancellation_policies[0].get("amount")
                    )
                    record["cancellation_from"] = (
                        cancellation_policies[0].get("from")
                    )
                else:
                    record["cancellation_amount"] = None
                    record["cancellation_from"] = None

                # Add the flattened record to our list.
                records.append(record)

    # Convert the collected records into a DataFrame.
    return pd.DataFrame(records)

In [14]:
# Convert the nested Goa API response into a flat DataFrame.
goa_accommodation_df = parse_hotelbeds_response(
    availability_response.json(),
    destination="Goa",
    check_in=check_in,
    check_out=check_out
)

# Display the resulting dataset dimensions.
print("Rows:", len(goa_accommodation_df))
print("Columns:", len(goa_accommodation_df.columns))

# Display the first few flattened accommodation records.
display(goa_accommodation_df.head())

Rows: 0
Columns: 0


""


In [27]:
display(goa_accommodation_df.head())

""


In [25]:
# Check the basic structure of the flattened accommodation dataset.
print("Rows:", len(goa_accommodation_df))
print("Columns:", len(goa_accommodation_df.columns))

# Check how many unique hotels were returned.
print("Unique hotels:", goa_accommodation_df["hotel_code"].nunique())

# Check how many unique rooms were returned.
print("Unique rooms:", goa_accommodation_df["room_code"].nunique())

# Check how many unique rates were returned.
print("Unique rates:", goa_accommodation_df["rate_key"].nunique())

# Check whether all returned rates are bookable.
print("\nRate types:")
print(goa_accommodation_df["rate_type"].value_counts())

# Check the price information.
print("\nPrice summary:")
print(goa_accommodation_df["price"].describe())

# Check missing values in important fields.
print("\nMissing important values:")
print(
    goa_accommodation_df[
        [
            "hotel_name",
            "room_name",
            "rate_key",
            "price",
            "board_name"
        ]
    ].isnull().sum()
)

Rows: 0
Columns: 0


KeyError: 'hotel_code'

In [ ]:
# Convert the price-related columns from text to numeric values.
# This is necessary because later we will calculate:
# - cheapest accommodation
# - average accommodation cost
# - total hotel cost
# - budget-based recommendations

goa_accommodation_df["price"] = pd.to_numeric(
    goa_accommodation_df["price"],
    errors="coerce"
)

goa_accommodation_df["offer_amount"] = pd.to_numeric(
    goa_accommodation_df["offer_amount"],
    errors="coerce"
)

goa_accommodation_df["cancellation_amount"] = pd.to_numeric(
    goa_accommodation_df["cancellation_amount"],
    errors="coerce"
)

# Convert date columns to proper datetime format.
goa_accommodation_df["check_in"] = pd.to_datetime(
    goa_accommodation_df["check_in"]
)

goa_accommodation_df["check_out"] = pd.to_datetime(
    goa_accommodation_df["check_out"]
)

# Check the resulting data types.
print(goa_accommodation_df[
    ["price", "offer_amount", "cancellation_amount",
     "check_in", "check_out"]
].dtypes)

price                         float64
offer_amount                  float64
cancellation_amount           float64
check_in               datetime64[us]
check_out              datetime64[us]
dtype: object


In [ ]:
# Check that price is now numeric.
print("Price data type:", goa_accommodation_df["price"].dtype)

# Check whether price conversion created missing values.
print("Missing prices:", goa_accommodation_df["price"].isna().sum())

# Display the cheapest available rates.
cheapest_rates = goa_accommodation_df.sort_values(
    "price"
)[
    [
        "hotel_name",
        "hotel_category_name",
        "room_name",
        "board_name",
        "price"
    ]
].head(10)

display(cheapest_rates)

Price data type: float64
Missing prices: 0


,hotel_name,hotel_category_name,room_name,board_name,price
241,Alagoa Resort,3 STARS,Classic,BED AND BREAKFAST,19.13
242,Alagoa Resort,3 STARS,Classic,ROOM ONLY,21.88
245,Alagoa Resort,3 STARS,Delux Room,ROOM ONLY,21.88
243,Alagoa Resort,3 STARS,Classic,BED AND BREAKFAST,24.54
246,Alagoa Resort,3 STARS,Delux Room,BED AND BREAKFAST,24.61
244,Alagoa Resort,3 STARS,Classic,ROOM ONLY,28.05
247,Alagoa Resort,3 STARS,Delux Room,ROOM ONLY,28.05
217,GINGER Goa Madgaon,3 STARS,Superior Double Single,ROOM ONLY,28.77
223,GINGER Goa Madgaon,3 STARS,Superior Twin Single,ROOM ONLY,28.77
42,Nanutel-Margao,3 STARS,Deluxe Double Room,ROOM ONLY,29.55


In [ ]:
# Create the accommodation output directory if it does not already exist.
import os

accommodation_dir = "../data/raw/accommodation"
os.makedirs(accommodation_dir, exist_ok=True)

# Save the flattened Goa accommodation data.
# This is our validated structured accommodation dataset.
goa_accommodation_path = (
    f"{accommodation_dir}/goa_accommodation_test.csv"
)

goa_accommodation_df.to_csv(
    goa_accommodation_path,
    index=False
)

print("Saved:", goa_accommodation_path)
print("Rows saved:", len(goa_accommodation_df))

Saved: ../data/raw/accommodation/goa_accommodation_test.csv
Rows saved: 249


In [ ]:
import json

# Save the original Hotelbeds response as JSON.
# Keeping the raw response allows us to reproduce or re-parse
# the data later if we decide to add more fields.
goa_raw_path = (
    f"{accommodation_dir}/goa_hotelbeds_raw.json"
)

with open(goa_raw_path, "w", encoding="utf-8") as file:
    json.dump(
        availability_response.json(),
        file,
        indent=2,
        ensure_ascii=False
    )

print("Raw API response saved:", goa_raw_path)

Raw API response saved: ../data/raw/accommodation/goa_hotelbeds_raw.json


# Accommodation Collection — 50 Destinations

The single-destination Hotelbeds test is now complete.

## What has been validated

- Hotelbeds authentication                 ✅
- Hotel availability request               ✅
- Hotel records                            ✅
- Room records                             ✅
- Bookable rates                           ✅
- Accommodation prices                    ✅
- Board information                        ✅
- Cancellation information                 ✅
- Response parser                          ✅
- Structured accommodation dataset         ✅

## Next Approach

We will now collect accommodation data for
all 50 destinations in `destination_master.csv`.

For every destination:

1. Read its latitude and longitude.
2. Send a Hotelbeds availability request.
3. Parse the response using the same parser.
4. Store the returned accommodation records.
5. Record whether the request succeeded or failed.
6. Keep the destination name attached to every record.

## Important API Limitation

We tested one destination first intentionally.

Before running the complete 50-destination collection,
we need to account for the Hotelbeds evaluation API
request limit.

Therefore, we will design the collection process so
that it can be resumed safely rather than repeatedly
requesting destinations that have already been collected.

## Final Target

After collection:

50 destinations
      ↓
Hotel availability
      ↓
Hotel + room + rate records
      ↓
Accommodation dataset
      ↓
Price features
      ↓
Trip cost calculation

# Collect Accommodation Data for 50 Destinations

The Goa accommodation test is complete.

We now move from testing to controlled collection.

## Collection Strategy

For each destination:

Destination Master
        ↓
Latitude + Longitude
        ↓
Hotelbeds Availability API
        ↓
Hotel + Room + Rate records
        ↓
Parser
        ↓
Append to accommodation dataset
        ↓
Save immediately

## Resumability

The collector will maintain two files:

1. `accommodation_raw.csv`
   - Successfully collected accommodation records.

2. `accommodation_collection_log.csv`
   - One record per destination.
   - Stores success/failure and number of records.

If the notebook stops halfway through the collection,
we can run it again.

Already completed destinations will be skipped.

## Important

We will NOT repeatedly request destinations that have
already been successfully collected.

This protects our API quota and prevents duplicate data.

In [ ]:
# Define where the accommodation collection files will be stored.
accommodation_dir = "../data/raw/accommodation"

os.makedirs(accommodation_dir, exist_ok=True)

# File containing all successfully collected accommodation records.
accommodation_data_path = (
    f"{accommodation_dir}/accommodation_raw.csv"
)

# File tracking the status of every destination request.
collection_log_path = (
    f"{accommodation_dir}/accommodation_collection_log.csv"
)

# If a previous collection exists, load it.
# Otherwise, start with an empty DataFrame using the parser's columns.
if os.path.exists(accommodation_data_path):

    accommodation_raw_df = pd.read_csv(
        accommodation_data_path
    )

    print("Existing accommodation data loaded.")

else:

    accommodation_raw_df = pd.DataFrame(
        columns=goa_accommodation_df.columns
    )

    print("No previous accommodation data found.")

No previous accommodation data found.


In [ ]:
# Create the collection log if it does not already exist.
#
# This allows us to know exactly which destinations have already
# been processed and whether each request succeeded or failed.

if os.path.exists(collection_log_path):

    accommodation_log_df = pd.read_csv(
        collection_log_path
    )

    print("Existing collection log loaded.")

else:

    accommodation_log_df = pd.DataFrame(
        columns=[
            "destination",
            "status",
            "records_returned",
            "error"
        ]
    )

    print("New collection log created.")

print(
    "Destinations already logged:",
    accommodation_log_df["destination"].nunique()
)

New collection log created.
Destinations already logged: 0


In [22]:
# Get the complete list of destinations from the master dataset.
all_destinations = destination_master[
    "destination"
].dropna().unique().tolist()

# Destinations that were successfully collected previously.
completed_destinations = set(
    accommodation_log_df.loc[
        accommodation_log_df["status"] == "success",
        "destination"
    ]
)

# Destinations that still need to be collected.
remaining_destinations = [
    destination
    for destination in all_destinations
    if destination not in completed_destinations
]

print("Total destinations:", len(all_destinations))
print("Completed:", len(completed_destinations))
print("Remaining:", len(remaining_destinations))

print("\nRemaining destinations:")
print(remaining_destinations)

Total destinations: 50
Completed: 0
Remaining: 50

Remaining destinations:
['Goa', 'Munnar', 'Manali', 'Jaipur', 'Udaipur', 'Jaisalmer', 'Jodhpur', 'Agra', 'Varanasi', 'Rishikesh', 'Shimla', 'Mussoorie', 'Nainital', 'Darjeeling', 'Gangtok', 'Ooty', 'Kodaikanal', 'Coorg', 'Wayanad', 'Alappuzha', 'Kochi', 'Thiruvananthapuram', 'Varkala', 'Pondicherry', 'Mahabalipuram', 'Hampi', 'Mysore', 'Gokarna', 'Andaman', 'Mumbai', 'Delhi', 'Amritsar', 'Ladakh', 'Srinagar', 'Dharamshala', 'Kolkata', 'Bengaluru', 'Hyderabad', 'Chennai', 'Pune', 'Ahmedabad', 'Bhopal', 'Indore', 'Ranchi', 'Bhubaneswar', 'Shillong', 'Kaziranga', 'Jim Corbett', 'Ranthambore', 'Pahalgam']


In [23]:
# ---------------------------------------------------------
# CONTROLLED BATCH 1
# ---------------------------------------------------------
# We intentionally collect only 5 destinations in this run.
# This protects the Hotelbeds daily quota and lets us verify
# the collected data before continuing with the remaining
# destinations.
# ---------------------------------------------------------

batch_size = 5

# Take only the first 5 destinations that have not been
# successfully collected yet.
current_batch = remaining_destinations[:batch_size]

print("Current batch size:", len(current_batch))
print("Destinations in this batch:")
print(current_batch)

Current batch size: 5
Destinations in this batch:
['Goa', 'Munnar', 'Manali', 'Jaipur', 'Udaipur']


In [24]:
# ---------------------------------------------------------
# COLLECT ACCOMMODATION DATA — BATCH 1
# ---------------------------------------------------------
# This cell sends one Hotelbeds request per destination
# for the 5 destinations selected in the previous cell.
#
# Progress is saved immediately after every destination,
# so we can safely stop and resume later.
# ---------------------------------------------------------

for destination in current_batch:

    print(f"\n{'=' * 60}")
    print(f"Collecting accommodation: {destination}")
    print(f"{'=' * 60}")

    try:
        # -----------------------------------------------------
        # 1. Find the validated coordinates for this destination.
        # -----------------------------------------------------
        location_match = destination_locations[
            destination_locations["destination"].str.lower()
            == destination.lower()
        ]

        if location_match.empty:
            raise ValueError(
                "Destination coordinates not found."
            )

        location = location_match.iloc[0]

        latitude = float(location["latitude"])
        longitude = float(location["longitude"])

        # -----------------------------------------------------
        # 2. Generate a fresh authentication signature.
        # -----------------------------------------------------
        timestamp = str(int(time.time()))

        signature_string = (
            HOTELBEDS_API_KEY
            + HOTELBEDS_SECRET
            + timestamp
        )

        signature = hashlib.sha256(
            signature_string.encode("utf-8")
        ).hexdigest()

        # -----------------------------------------------------
        # 3. Create the Hotelbeds availability request.
        # -----------------------------------------------------
        request_body = {
            "stay": {
                "checkIn": check_in,
                "checkOut": check_out
            },
            "occupancies": [
                {
                    "rooms": 1,
                    "adults": 1,
                    "children": 0
                }
            ],
            "geolocation": {
                "latitude": latitude,
                "longitude": longitude,
                "radius": 20,
                "unit": "km"
            }
        }

        request_headers = {
            "Accept": "application/json",
            "Content-Type": "application/json",
            "Api-key": HOTELBEDS_API_KEY,
            "X-Signature": signature
        }

        # -----------------------------------------------------
        # 4. Send the API request.
        # -----------------------------------------------------
        response = requests.post(
            availability_url,
            headers=request_headers,
            json=request_body,
            timeout=60
        )

        response.raise_for_status()

        response_json = response.json()

        # -----------------------------------------------------
        # 5. Flatten the Hotelbeds response.
        # -----------------------------------------------------
        destination_accommodation = parse_hotelbeds_response(
            response_json,
            destination=destination,
            check_in=check_in,
            check_out=check_out
        )

        records_count = len(destination_accommodation)

        # -----------------------------------------------------
        # 6. Add records to our master accommodation dataset.
        # -----------------------------------------------------
        if records_count > 0:

            accommodation_raw_df = pd.concat(
                [
                    accommodation_raw_df,
                    destination_accommodation
                ],
                ignore_index=True
            )

        # -----------------------------------------------------
        # 7. Record successful collection.
        # -----------------------------------------------------
        new_log = pd.DataFrame([
            {
                "destination": destination,
                "status": "success",
                "records_returned": records_count,
                "error": None
            }
        ])

        accommodation_log_df = pd.concat(
            [
                accommodation_log_df,
                new_log
            ],
            ignore_index=True
        )

        # -----------------------------------------------------
        # 8. Save immediately.
        # -----------------------------------------------------
        accommodation_raw_df.to_csv(
            accommodation_data_path,
            index=False
        )

        accommodation_log_df.to_csv(
            collection_log_path,
            index=False
        )

        print(
            f"Success → {records_count} accommodation records"
        )

        # Wait before the next request.
        # This is well below the API's allowed request rate.
        time.sleep(1)

    except Exception as e:

        # -----------------------------------------------------
        # Record failures without stopping the entire batch.
        # -----------------------------------------------------
        error_message = str(e)

        new_log = pd.DataFrame([
            {
                "destination": destination,
                "status": "failed",
                "records_returned": 0,
                "error": error_message
            }
        ])

        accommodation_log_df = pd.concat(
            [
                accommodation_log_df,
                new_log
            ],
            ignore_index=True
        )

        accommodation_log_df.to_csv(
            collection_log_path,
            index=False
        )

        print(f"FAILED → {destination}")
        print("Reason:", error_message)

print("\nBatch 1 collection completed.")


Success → 249 accommodation records

Success → 15 accommodation records

Success → 48 accommodation records

Success → 492 accommodation records

Success → 153 accommodation records

Batch 1 collection completed.


In [25]:
# ---------------------------------------------------------
# VALIDATE BATCH 1
# ---------------------------------------------------------
# We verify that:
# 1. All 5 destinations were recorded as successful.
# 2. No destination was duplicated in the collection log.
# 3. Accommodation records exist for each destination.
# 4. Important fields are not missing.
# ---------------------------------------------------------

batch_log = accommodation_log_df[
    accommodation_log_df["destination"].isin(current_batch)
]

print("Batch destinations:", len(current_batch))
print("Log records:", len(batch_log))

print("\nCollection status:")
display(
    batch_log[
        [
            "destination",
            "status",
            "records_returned"
        ]
    ]
)

# Check for duplicate destination entries in the log.
print(
    "\nDuplicate log destinations:",
    batch_log["destination"].duplicated().sum()
)

# Get accommodation records belonging to this batch.
batch_accommodation = accommodation_raw_df[
    accommodation_raw_df["destination"].isin(current_batch)
]

print(
    "\nAccommodation records:",
    len(batch_accommodation)
)

print(
    "Unique destinations in accommodation data:",
    batch_accommodation["destination"].nunique()
)

# Check important fields for missing values.
important_columns = [
    "destination",
    "hotel_code",
    "hotel_name",
    "room_code",
    "room_name",
    "rate_key",
    "price",
    "board_name"
]

print("\nMissing important values:")
print(
    batch_accommodation[important_columns].isna().sum()
)

Batch destinations: 5
Log records: 5

Collection status:


,destination,status,records_returned
0,Goa,success,249
1,Munnar,success,15
2,Manali,success,48
3,Jaipur,success,492
4,Udaipur,success,153



Duplicate log destinations: 0

Accommodation records: 957
Unique destinations in accommodation data: 5

Missing important values:
destination    0
hotel_code     0
hotel_name     0
room_code      0
room_name      0
rate_key       0
price          0
board_name     0
dtype: int64


# Accommodation Collection — Batch 2

Batch 1 has been successfully collected and validated.

### Batch 1
- Goa
- Munnar
- Manali
- Jaipur
- Udaipur

### Result
- 5 destinations completed
- 957 accommodation rate records
- No duplicate destination logs
- No missing important accommodation fields

---

## Next Batch

We will now collect the next 5 destinations:

1. Jaisalmer
2. Jodhpur
3. Agra
4. Varanasi
5. Rishikesh

The same validated Hotelbeds parser and collection process
will be reused.

### Flow

Destination coordinates
        ↓
Hotelbeds availability API
        ↓
Hotel + Room + Rate
        ↓
Flatten response
        ↓
Append to accommodation_raw.csv
        ↓
Update collection log
        ↓
Validate batch

We will continue in batches rather than sending all
remaining requests at once.

In [26]:
# ---------------------------------------------------------
# SELECT BATCH 2
# ---------------------------------------------------------
# Recalculate completed destinations from the collection log.
# This makes the notebook resumable even if it is restarted.
# ---------------------------------------------------------

completed_destinations = set(
    accommodation_log_df.loc[
        accommodation_log_df["status"] == "success",
        "destination"
    ]
)

# Find destinations that still need accommodation collection.
remaining_destinations = [
    destination
    for destination in all_destinations
    if destination not in completed_destinations
]

# Select the next 5 destinations.
batch_size = 5
current_batch = remaining_destinations[:batch_size]

print("Completed destinations:", len(completed_destinations))
print("Remaining destinations:", len(remaining_destinations))

print("\nBatch 2:")
print(current_batch)

Completed destinations: 5
Remaining destinations: 45

Batch 2:
['Jaisalmer', 'Jodhpur', 'Agra', 'Varanasi', 'Rishikesh']


In [27]:
# ---------------------------------------------------------
# COLLECT ACCOMMODATION DATA — BATCH 2
# ---------------------------------------------------------
# We process only the 5 destinations selected in Cell 34.
# Every successful destination is saved immediately so that
# the collection can be safely resumed later.
# ---------------------------------------------------------

for destination in current_batch:

    print(f"\n{'=' * 60}")
    print(f"Collecting accommodation: {destination}")
    print(f"{'=' * 60}")

    try:
        # Find the validated coordinates for this destination.
        location_match = destination_locations[
            destination_locations["destination"].str.lower()
            == destination.lower()
        ]

        if location_match.empty:
            raise ValueError("Destination coordinates not found.")

        location = location_match.iloc[0]

        latitude = float(location["latitude"])
        longitude = float(location["longitude"])

        # Generate a fresh Hotelbeds authentication signature.
        timestamp = str(int(time.time()))

        signature_string = (
            HOTELBEDS_API_KEY
            + HOTELBEDS_SECRET
            + timestamp
        )

        signature = hashlib.sha256(
            signature_string.encode("utf-8")
        ).hexdigest()

        # Prepare the hotel availability request.
        request_body = {
            "stay": {
                "checkIn": check_in,
                "checkOut": check_out
            },
            "occupancies": [
                {
                    "rooms": 1,
                    "adults": 1,
                    "children": 0
                }
            ],
            "geolocation": {
                "latitude": latitude,
                "longitude": longitude,
                "radius": 20,
                "unit": "km"
            }
        }

        request_headers = {
            "Accept": "application/json",
            "Content-Type": "application/json",
            "Api-key": HOTELBEDS_API_KEY,
            "X-Signature": signature
        }

        # Send the request.
        response = requests.post(
            availability_url,
            headers=request_headers,
            json=request_body,
            timeout=60
        )

        response.raise_for_status()

        # Parse the nested API response into our flat structure.
        destination_accommodation = parse_hotelbeds_response(
            response.json(),
            destination=destination,
            check_in=check_in,
            check_out=check_out
        )

        records_count = len(destination_accommodation)

        # Add returned records to the master accommodation dataset.
        if records_count > 0:

            accommodation_raw_df = pd.concat(
                [
                    accommodation_raw_df,
                    destination_accommodation
                ],
                ignore_index=True
            )

        # Record successful collection.
        new_log = pd.DataFrame([
            {
                "destination": destination,
                "status": "success",
                "records_returned": records_count,
                "error": None
            }
        ])

        accommodation_log_df = pd.concat(
            [
                accommodation_log_df,
                new_log
            ],
            ignore_index=True
        )

        # Save immediately after each successful destination.
        accommodation_raw_df.to_csv(
            accommodation_data_path,
            index=False
        )

        accommodation_log_df.to_csv(
            collection_log_path,
            index=False
        )

        print(f"Success → {records_count} accommodation records")

        # Small delay between requests.
        time.sleep(1)

    except Exception as e:

        # Record the failure without stopping the batch.
        error_message = str(e)

        new_log = pd.DataFrame([
            {
                "destination": destination,
                "status": "failed",
                "records_returned": 0,
                "error": error_message
            }
        ])

        accommodation_log_df = pd.concat(
            [
                accommodation_log_df,
                new_log
            ],
            ignore_index=True
        )

        accommodation_log_df.to_csv(
            collection_log_path,
            index=False
        )

        print(f"FAILED → {destination}")
        print("Reason:", error_message)

print("\nBatch 2 collection completed.")


Success → 24 accommodation records

Success → 35 accommodation records

Success → 44 accommodation records

Success → 64 accommodation records

Success → 126 accommodation records

Batch 2 collection completed.


In [30]:
# ---------------------------------------------------------
# VALIDATE BATCH 2
# ---------------------------------------------------------
# Confirm that all five destinations were collected once,
# accommodation records exist, and important fields are complete.
# ---------------------------------------------------------

batch_log = accommodation_log_df[
    accommodation_log_df["destination"].isin(current_batch)
]

print("Batch destinations:", len(current_batch))
print("Log records:", len(batch_log))

print("\nCollection status:")
display(
    batch_log[
        [
            "destination",
            "status",
            "records_returned"
        ]
    ]
)

# Check for duplicate destination entries.
print(
    "\nDuplicate log destinations:",
    batch_log["destination"].duplicated().sum()
)

# Get accommodation records for this batch.
batch_accommodation = accommodation_raw_df[
    accommodation_raw_df["destination"].isin(current_batch)
]

print(
    "\nAccommodation records:",
    len(batch_accommodation)
)

print(
    "Unique destinations:",
    batch_accommodation["destination"].nunique()
)

# Check important fields for missing values.
important_columns = [
    "destination",
    "hotel_code",
    "hotel_name",
    "room_code",
    "room_name",
    "rate_key",
    "price",
    "board_name"
]

print("\nMissing important values:")
print(
    batch_accommodation[important_columns].isna().sum()
)

Batch destinations: 5
Log records: 5

Collection status:


,destination,status,records_returned
5,Jaisalmer,success,24
6,Jodhpur,success,35
7,Agra,success,44
8,Varanasi,success,64
9,Rishikesh,success,126



Duplicate log destinations: 0

Accommodation records: 293
Unique destinations: 5

Missing important values:
destination    0
hotel_code     0
hotel_name     0
room_code      0
room_name      0
rate_key       0
price          0
board_name     0
dtype: int64


# Accommodation Collection — Batch 3

Batch 2 has been successfully collected and validated.

### Completed so far

- Batch 1: 5 destinations
- Batch 2: 5 destinations
- Total completed: 10 / 50
- Total accommodation records: 1,250

---

## Next Batch

We will collect the next 5 destinations:

1. Shimla
2. Mussoorie
3. Nainital
4. Darjeeling
5. Gangtok

We will use the same validated process:

Destination
    ↓
Coordinates
    ↓
Hotelbeds API
    ↓
Parse hotel → room → rate
    ↓
Append to accommodation dataset
    ↓
Save
    ↓
Validate

In [31]:
# ---------------------------------------------------------
# SELECT BATCH 3
# ---------------------------------------------------------
# Recalculate completed destinations from the saved log.
# This keeps the process resumable and prevents duplicate
# API requests if the notebook is restarted.
# ---------------------------------------------------------

completed_destinations = set(
    accommodation_log_df.loc[
        accommodation_log_df["status"] == "success",
        "destination"
    ]
)

# Find destinations that are still not successfully collected.
remaining_destinations = [
    destination
    for destination in all_destinations
    if destination not in completed_destinations
]

# Select the next 5 destinations.
batch_size = 5
current_batch = remaining_destinations[:batch_size]

print("Completed destinations:", len(completed_destinations))
print("Remaining destinations:", len(remaining_destinations))

print("\nBatch 3:")
print(current_batch)

Completed destinations: 10
Remaining destinations: 40

Batch 3:
['Shimla', 'Mussoorie', 'Nainital', 'Darjeeling', 'Gangtok']


In [32]:
# ---------------------------------------------------------
# COLLECT ACCOMMODATION DATA — BATCH 3
# ---------------------------------------------------------
# Collect only the 5 destinations selected in current_batch.
# The data and collection log are saved after every
# destination so the process can be resumed safely.
# ---------------------------------------------------------

for destination in current_batch:

    print(f"\n{'=' * 60}")
    print(f"Collecting accommodation: {destination}")
    print(f"{'=' * 60}")

    try:
        # Find validated latitude and longitude.
        location_match = destination_locations[
            destination_locations["destination"].str.lower()
            == destination.lower()
        ]

        if location_match.empty:
            raise ValueError("Destination coordinates not found.")

        location = location_match.iloc[0]

        latitude = float(location["latitude"])
        longitude = float(location["longitude"])

        # Generate a fresh Hotelbeds signature.
        timestamp = str(int(time.time()))

        signature_string = (
            HOTELBEDS_API_KEY
            + HOTELBEDS_SECRET
            + timestamp
        )

        signature = hashlib.sha256(
            signature_string.encode("utf-8")
        ).hexdigest()

        # Prepare the availability request.
        request_body = {
            "stay": {
                "checkIn": check_in,
                "checkOut": check_out
            },
            "occupancies": [
                {
                    "rooms": 1,
                    "adults": 1,
                    "children": 0
                }
            ],
            "geolocation": {
                "latitude": latitude,
                "longitude": longitude,
                "radius": 20,
                "unit": "km"
            }
        }

        request_headers = {
            "Accept": "application/json",
            "Content-Type": "application/json",
            "Api-key": HOTELBEDS_API_KEY,
            "X-Signature": signature
        }

        # Send the Hotelbeds request.
        response = requests.post(
            availability_url,
            headers=request_headers,
            json=request_body,
            timeout=60
        )

        response.raise_for_status()

        # Parse the nested response into our flat dataset.
        destination_accommodation = parse_hotelbeds_response(
            response.json(),
            destination=destination,
            check_in=check_in,
            check_out=check_out
        )

        records_count = len(destination_accommodation)

        # Add the returned records to the master dataset.
        if records_count > 0:
            accommodation_raw_df = pd.concat(
                [
                    accommodation_raw_df,
                    destination_accommodation
                ],
                ignore_index=True
            )

        # Record successful collection.
        new_log = pd.DataFrame([
            {
                "destination": destination,
                "status": "success",
                "records_returned": records_count,
                "error": None
            }
        ])

        accommodation_log_df = pd.concat(
            [
                accommodation_log_df,
                new_log
            ],
            ignore_index=True
        )

        # Save immediately after each destination.
        accommodation_raw_df.to_csv(
            accommodation_data_path,
            index=False
        )

        accommodation_log_df.to_csv(
            collection_log_path,
            index=False
        )

        print(
            f"Success → {records_count} accommodation records"
        )

        # Pause briefly between requests.
        time.sleep(1)

    except Exception as e:

        # Record the failure and continue with the next destination.
        error_message = str(e)

        new_log = pd.DataFrame([
            {
                "destination": destination,
                "status": "failed",
                "records_returned": 0,
                "error": error_message
            }
        ])

        accommodation_log_df = pd.concat(
            [
                accommodation_log_df,
                new_log
            ],
            ignore_index=True
        )

        accommodation_log_df.to_csv(
            collection_log_path,
            index=False
        )

        print(f"FAILED → {destination}")
        print("Reason:", error_message)

print("\nBatch 3 collection completed.")


Success → 41 accommodation records

Success → 42 accommodation records

Success → 87 accommodation records

Success → 93 accommodation records

Success → 146 accommodation records

Batch 3 collection completed.


In [33]:
# ---------------------------------------------------------
# VALIDATE BATCH 3
# ---------------------------------------------------------
# Confirm that all five destinations were collected exactly
# once and that the important accommodation fields are complete.
# ---------------------------------------------------------

batch_log = accommodation_log_df[
    accommodation_log_df["destination"].isin(current_batch)
]

print("Batch destinations:", len(current_batch))
print("Log records:", len(batch_log))

print("\nCollection status:")
display(
    batch_log[
        [
            "destination",
            "status",
            "records_returned"
        ]
    ]
)

# Check for duplicate destination entries.
print(
    "\nDuplicate log destinations:",
    batch_log["destination"].duplicated().sum()
)

# Get accommodation records for Batch 3.
batch_accommodation = accommodation_raw_df[
    accommodation_raw_df["destination"].isin(current_batch)
]

print(
    "\nAccommodation records:",
    len(batch_accommodation)
)

print(
    "Unique destinations:",
    batch_accommodation["destination"].nunique()
)

# Check important fields for missing values.
important_columns = [
    "destination",
    "hotel_code",
    "hotel_name",
    "room_code",
    "room_name",
    "rate_key",
    "price",
    "board_name"
]

print("\nMissing important values:")
print(
    batch_accommodation[important_columns].isna().sum()
)

Batch destinations: 5
Log records: 5

Collection status:


,destination,status,records_returned
10,Shimla,success,41
11,Mussoorie,success,42
12,Nainital,success,87
13,Darjeeling,success,93
14,Gangtok,success,146



Duplicate log destinations: 0

Accommodation records: 409
Unique destinations: 5

Missing important values:
destination    0
hotel_code     0
hotel_name     0
room_code      0
room_name      0
rate_key       0
price          0
board_name     0
dtype: int64


In [34]:
# ---------------------------------------------------------
# SELECT BATCH 4
# ---------------------------------------------------------
# Recalculate progress from the saved collection log.
# This prevents already-completed destinations from being
# requested again.
# ---------------------------------------------------------

completed_destinations = set(
    accommodation_log_df.loc[
        accommodation_log_df["status"] == "success",
        "destination"
    ]
)

# Find destinations still waiting for accommodation data.
remaining_destinations = [
    destination
    for destination in all_destinations
    if destination not in completed_destinations
]

# Select the next 5 destinations.
batch_size = 10
current_batch = remaining_destinations[:batch_size]

print("Completed destinations:", len(completed_destinations))
print("Remaining destinations:", len(remaining_destinations))

print("\nBatch 4:")
print(current_batch)

Completed destinations: 15
Remaining destinations: 35

Batch 4:
['Ooty', 'Kodaikanal', 'Coorg', 'Wayanad', 'Alappuzha', 'Kochi', 'Thiruvananthapuram', 'Varkala', 'Pondicherry', 'Mahabalipuram']


In [35]:
# ---------------------------------------------------------
# COLLECT ACCOMMODATION DATA — BATCH 4
# ---------------------------------------------------------
# Collect the 10 destinations selected above.
# Every destination is saved immediately after processing.
# ---------------------------------------------------------

for destination in current_batch:

    print(f"\n{'=' * 60}")
    print(f"Collecting accommodation: {destination}")
    print(f"{'=' * 60}")

    try:
        # Find validated coordinates.
        location_match = destination_locations[
            destination_locations["destination"].str.lower()
            == destination.lower()
        ]

        if location_match.empty:
            raise ValueError("Destination coordinates not found.")

        location = location_match.iloc[0]

        latitude = float(location["latitude"])
        longitude = float(location["longitude"])

        # Create a fresh authentication signature.
        timestamp = str(int(time.time()))

        signature_string = (
            HOTELBEDS_API_KEY
            + HOTELBEDS_SECRET
            + timestamp
        )

        signature = hashlib.sha256(
            signature_string.encode("utf-8")
        ).hexdigest()

        # Prepare Hotelbeds request.
        request_body = {
            "stay": {
                "checkIn": check_in,
                "checkOut": check_out
            },
            "occupancies": [
                {
                    "rooms": 1,
                    "adults": 1,
                    "children": 0
                }
            ],
            "geolocation": {
                "latitude": latitude,
                "longitude": longitude,
                "radius": 20,
                "unit": "km"
            }
        }

        request_headers = {
            "Accept": "application/json",
            "Content-Type": "application/json",
            "Api-key": HOTELBEDS_API_KEY,
            "X-Signature": signature
        }

        # Send request.
        response = requests.post(
            availability_url,
            headers=request_headers,
            json=request_body,
            timeout=60
        )

        response.raise_for_status()

        # Parse nested Hotelbeds response.
        destination_accommodation = parse_hotelbeds_response(
            response.json(),
            destination=destination,
            check_in=check_in,
            check_out=check_out
        )

        records_count = len(destination_accommodation)

        # Add records to master dataset.
        if records_count > 0:
            accommodation_raw_df = pd.concat(
                [
                    accommodation_raw_df,
                    destination_accommodation
                ],
                ignore_index=True
            )

        # Record collection status.
        new_log = pd.DataFrame([
            {
                "destination": destination,
                "status": "success",
                "records_returned": records_count,
                "error": None
            }
        ])

        accommodation_log_df = pd.concat(
            [
                accommodation_log_df,
                new_log
            ],
            ignore_index=True
        )

        # Save immediately.
        accommodation_raw_df.to_csv(
            accommodation_data_path,
            index=False
        )

        accommodation_log_df.to_csv(
            collection_log_path,
            index=False
        )

        print(
            f"Success → {records_count} accommodation records"
        )

        # Keep a small delay between requests.
        time.sleep(1)

    except Exception as e:

        # Record failures without stopping the batch.
        error_message = str(e)

        new_log = pd.DataFrame([
            {
                "destination": destination,
                "status": "failed",
                "records_returned": 0,
                "error": error_message
            }
        ])

        accommodation_log_df = pd.concat(
            [
                accommodation_log_df,
                new_log
            ],
            ignore_index=True
        )

        accommodation_log_df.to_csv(
            collection_log_path,
            index=False
        )

        print(f"FAILED → {destination}")
        print("Reason:", error_message)

print("\nBatch 4 collection completed.")


Success → 29 accommodation records

Success → 0 accommodation records

Success → 0 accommodation records

Success → 0 accommodation records

Success → 9 accommodation records

Success → 89 accommodation records

Success → 68 accommodation records

Success → 4 accommodation records

Success → 38 accommodation records

Success → 0 accommodation records

Batch 4 collection completed.


In [36]:
# ---------------------------------------------------------
# VALIDATE BATCH 4
# ---------------------------------------------------------
# We check the collection log and separate:
# 1. Successful destinations with accommodation records
# 2. Successful API responses with zero records
# 3. Actual failed API requests
# ---------------------------------------------------------

batch_log = accommodation_log_df[
    accommodation_log_df["destination"].isin(current_batch)
].copy()

print("Batch destinations:", len(current_batch))
print("Log records:", len(batch_log))

print("\nCollection status:")
display(
    batch_log[
        [
            "destination",
            "status",
            "records_returned",
            "error"
        ]
    ]
)

# Destinations where the API request succeeded but returned
# zero accommodation records.
zero_record_destinations = batch_log[
    (batch_log["status"] == "success") &
    (batch_log["records_returned"] == 0)
]["destination"].tolist()

print("\nSuccessful but zero accommodation:")
print(zero_record_destinations)

# Destinations where the API request itself failed.
failed_batch_destinations = batch_log[
    batch_log["status"] == "failed"
]["destination"].tolist()

print("\nActual API failures:")
print(failed_batch_destinations)

# Get records returned for this batch.
batch_accommodation = accommodation_raw_df[
    accommodation_raw_df["destination"].isin(current_batch)
]

print("\nAccommodation records:", len(batch_accommodation))

print(
    "Unique destinations with records:",
    batch_accommodation["destination"].nunique()
)

# Check important fields.
important_columns = [
    "destination",
    "hotel_code",
    "hotel_name",
    "room_code",
    "room_name",
    "rate_key",
    "price",
    "board_name"
]

print("\nMissing important values:")
print(
    batch_accommodation[important_columns].isna().sum()
)

Batch destinations: 10
Log records: 10

Collection status:


,destination,status,records_returned,error
15,Ooty,success,29,None
16,Kodaikanal,success,0,None
17,Coorg,success,0,None
18,Wayanad,success,0,None
19,Alappuzha,success,9,None
20,Kochi,success,89,None
21,Thiruvananthapuram,success,68,None
22,Varkala,success,4,None
23,Pondicherry,success,38,None
24,Mahabalipuram,success,0,None



Successful but zero accommodation:
['Kodaikanal', 'Coorg', 'Wayanad', 'Mahabalipuram']

Actual API failures:
[]

Accommodation records: 237
Unique destinations with records: 6

Missing important values:
destination    0
hotel_code     0
hotel_name     0
room_code      0
room_name      0
rate_key       0
price          0
board_name     0
dtype: int64


# Accommodation Availability Status

Some destinations returned accommodation records, while some
successfully returned zero records.

A zero-record response is NOT an API failure.

We will classify every destination as:

- AVAILABLE → accommodation records were returned
- NO_AVAILABILITY → API succeeded but returned zero records
- API_FAILED → the API request itself failed

This distinction is important for the final Travel Agent.

If a destination has NO_AVAILABILITY, the final system should
not claim that the destination has no hotels. It should instead
recognize that no matching Hotelbeds availability was returned
for our current search parameters.

In [37]:
# ---------------------------------------------------------
# ADD AVAILABILITY STATUS
# ---------------------------------------------------------
# Convert the existing collection results into a clearer
# availability classification for the final Travel Agent.
# ---------------------------------------------------------

def get_availability_status(row):
    """
    Classify a destination based on the API result.

    AVAILABLE:
        API succeeded and accommodation records were returned.

    NO_AVAILABILITY:
        API succeeded but returned zero records.

    API_FAILED:
        The API request itself failed.
    """

    if row["status"] == "failed":
        return "API_FAILED"

    if row["records_returned"] == 0:
        return "NO_AVAILABILITY"

    return "AVAILABLE"


# Apply the classification to every collection-log record.
accommodation_log_df["availability_status"] = (
    accommodation_log_df.apply(
        get_availability_status,
        axis=1
    )
)

# Save the updated log.
accommodation_log_df.to_csv(
    collection_log_path,
    index=False
)

# Display the current summary.
print(
    accommodation_log_df[
        "availability_status"
    ].value_counts()
)

availability_status
AVAILABLE          21
NO_AVAILABILITY     4
Name: count, dtype: int64


# Accommodation Collection — Batch 5

We have now processed 25 of the 50 destinations.

### Progress

Completed: 25 / 50
Remaining: 25

The accommodation collection process is working correctly.

We will now collect the next 10 destinations.

## Batch 5

1. Hampi
2. Mysore
3. Gokarna
4. Andaman
5. Mumbai
6. Delhi
7. Amritsar
8. Ladakh
9. Srinagar
10. Dharamshala

The same validated process will be used.

Destination
    ↓
Coordinates
    ↓
Hotelbeds API
    ↓
Hotel + Room + Rate
    ↓
Flatten
    ↓
Save
    ↓
Availability classification

In [39]:
# ---------------------------------------------------------
# SELECT BATCH 5
# ---------------------------------------------------------
# Recalculate completed destinations from the saved log.
# This makes the process safe to resume.
# ---------------------------------------------------------

completed_destinations = set(
    accommodation_log_df.loc[
        accommodation_log_df["status"].isin(
            ["success"]
        ),
        "destination"
    ]
)

# Find destinations that have not yet been processed.
remaining_destinations = [
    destination
    for destination in all_destinations
    if destination not in completed_destinations
]

# Select the next 10 destinations.
batch_size = 10
current_batch = remaining_destinations[:batch_size]

print("Processed destinations:", len(completed_destinations))
print("Remaining destinations:", len(remaining_destinations))

print("\nBatch 5 size:", len(current_batch))

for i, destination in enumerate(current_batch, start=1):
    print(f"{i}. {destination}")

Processed destinations: 25
Remaining destinations: 25

Batch 5 size: 10
1. Hampi
2. Mysore
3. Gokarna
4. Andaman
5. Mumbai
6. Delhi
7. Amritsar
8. Ladakh
9. Srinagar
10. Dharamshala


In [40]:
# ---------------------------------------------------------
# COLLECT ACCOMMODATION DATA — BATCH 5
# ---------------------------------------------------------
# Collect accommodation availability for the 10 destinations
# in current_batch.
#
# Every destination is saved immediately after processing,
# so our progress is protected if the notebook stops.
# ---------------------------------------------------------

for destination in current_batch:

    print(f"\n{'=' * 60}")
    print(f"Collecting accommodation: {destination}")
    print(f"{'=' * 60}")

    try:
        # -----------------------------------------------------
        # 1. Get validated coordinates for the destination.
        # -----------------------------------------------------
        location_match = destination_locations[
            destination_locations["destination"].str.lower()
            == destination.lower()
        ]

        if location_match.empty:
            raise ValueError("Destination coordinates not found.")

        location = location_match.iloc[0]

        latitude = float(location["latitude"])
        longitude = float(location["longitude"])

        # -----------------------------------------------------
        # 2. Generate Hotelbeds authentication signature.
        # -----------------------------------------------------
        timestamp = str(int(time.time()))

        signature_string = (
            HOTELBEDS_API_KEY
            + HOTELBEDS_SECRET
            + timestamp
        )

        signature = hashlib.sha256(
            signature_string.encode("utf-8")
        ).hexdigest()

        # -----------------------------------------------------
        # 3. Prepare availability request.
        # -----------------------------------------------------
        request_body = {
            "stay": {
                "checkIn": check_in,
                "checkOut": check_out
            },
            "occupancies": [
                {
                    "rooms": 1,
                    "adults": 1,
                    "children": 0
                }
            ],
            "geolocation": {
                "latitude": latitude,
                "longitude": longitude,
                "radius": 20,
                "unit": "km"
            }
        }

        request_headers = {
            "Accept": "application/json",
            "Content-Type": "application/json",
            "Api-key": HOTELBEDS_API_KEY,
            "X-Signature": signature
        }

        # -----------------------------------------------------
        # 4. Send request to Hotelbeds.
        # -----------------------------------------------------
        response = requests.post(
            availability_url,
            headers=request_headers,
            json=request_body,
            timeout=60
        )

        response.raise_for_status()

        # -----------------------------------------------------
        # 5. Parse the nested Hotelbeds response.
        # -----------------------------------------------------
        destination_accommodation = parse_hotelbeds_response(
            response.json(),
            destination=destination,
            check_in=check_in,
            check_out=check_out
        )

        records_count = len(destination_accommodation)

        # -----------------------------------------------------
        # 6. Append returned records.
        # -----------------------------------------------------
        if records_count > 0:

            accommodation_raw_df = pd.concat(
                [
                    accommodation_raw_df,
                    destination_accommodation
                ],
                ignore_index=True
            )

        # -----------------------------------------------------
        # 7. Determine availability status.
        # -----------------------------------------------------
        if records_count > 0:
            availability_status = "AVAILABLE"
        else:
            availability_status = "NO_AVAILABILITY"

        # -----------------------------------------------------
        # 8. Record collection result.
        # -----------------------------------------------------
        new_log = pd.DataFrame([
            {
                "destination": destination,
                "status": "success",
                "records_returned": records_count,
                "error": None,
                "availability_status": availability_status
            }
        ])

        accommodation_log_df = pd.concat(
            [
                accommodation_log_df,
                new_log
            ],
            ignore_index=True
        )

        # -----------------------------------------------------
        # 9. Save immediately.
        # -----------------------------------------------------
        accommodation_raw_df.to_csv(
            accommodation_data_path,
            index=False
        )

        accommodation_log_df.to_csv(
            collection_log_path,
            index=False
        )

        print(
            f"Success → {records_count} records "
            f"({availability_status})"
        )

        # Small delay between requests.
        time.sleep(1)

    except Exception as e:

        # -----------------------------------------------------
        # 10. Record API failures separately.
        # -----------------------------------------------------
        error_message = str(e)

        new_log = pd.DataFrame([
            {
                "destination": destination,
                "status": "failed",
                "records_returned": 0,
                "error": error_message,
                "availability_status": "API_FAILED"
            }
        ])

        accommodation_log_df = pd.concat(
            [
                accommodation_log_df,
                new_log
            ],
            ignore_index=True
        )

        accommodation_log_df.to_csv(
            collection_log_path,
            index=False
        )

        print(f"FAILED → {destination}")
        print("Reason:", error_message)

print("\nBatch 5 collection completed.")


Success → 20 records (AVAILABLE)

Success → 44 records (AVAILABLE)

Success → 10 records (AVAILABLE)

Success → 0 records (NO_AVAILABILITY)

Success → 491 records (AVAILABLE)

Success → 582 records (AVAILABLE)

Success → 66 records (AVAILABLE)

Success → 0 records (NO_AVAILABILITY)

Success → 3 records (AVAILABLE)

Success → 34 records (AVAILABLE)

Batch 5 collection completed.


In [41]:
# ---------------------------------------------------------
# VALIDATE BATCH 5
# ---------------------------------------------------------
# Verify that every destination in Batch 5 was processed,
# identify available vs zero-result destinations, and check
# that the returned accommodation records contain no missing
# important fields.
# ---------------------------------------------------------

batch_log = accommodation_log_df[
    accommodation_log_df["destination"].isin(current_batch)
].copy()

print("Batch destinations:", len(current_batch))
print("Log records:", len(batch_log))

print("\nCollection status:")
display(
    batch_log[
        [
            "destination",
            "status",
            "records_returned",
            "availability_status"
        ]
    ]
)

# Check whether any destination was logged more than once.
print(
    "\nDuplicate log destinations:",
    batch_log["destination"].duplicated().sum()
)

# Identify destinations where the API succeeded but returned
# no accommodation records.
zero_record_destinations = batch_log[
    batch_log["availability_status"] == "NO_AVAILABILITY"
]["destination"].tolist()

print("\nNo availability:")
print(zero_record_destinations)

# Identify actual API failures.
api_failures = batch_log[
    batch_log["availability_status"] == "API_FAILED"
]["destination"].tolist()

print("\nAPI failures:")
print(api_failures)

# Get accommodation records belonging to Batch 5.
batch_accommodation = accommodation_raw_df[
    accommodation_raw_df["destination"].isin(current_batch)
]

print(
    "\nAccommodation records:",
    len(batch_accommodation)
)

print(
    "Unique destinations with accommodation:",
    batch_accommodation["destination"].nunique()
)

# Check the important fields required for later cost calculations
# and accommodation recommendations.
important_columns = [
    "destination",
    "hotel_code",
    "hotel_name",
    "room_code",
    "room_name",
    "rate_key",
    "price",
    "board_name"
]

print("\nMissing important values:")
print(
    batch_accommodation[important_columns].isna().sum()
)

Batch destinations: 10
Log records: 10

Collection status:


,destination,status,records_returned,availability_status
25,Hampi,success,20,AVAILABLE
26,Mysore,success,44,AVAILABLE
27,Gokarna,success,10,AVAILABLE
28,Andaman,success,0,NO_AVAILABILITY
29,Mumbai,success,491,AVAILABLE
30,Delhi,success,582,AVAILABLE
31,Amritsar,success,66,AVAILABLE
32,Ladakh,success,0,NO_AVAILABILITY
33,Srinagar,success,3,AVAILABLE
34,Dharamshala,success,34,AVAILABLE



Duplicate log destinations: 0

No availability:
['Andaman', 'Ladakh']

API failures:
[]

Accommodation records: 1250
Unique destinations with accommodation: 8

Missing important values:
destination    0
hotel_code     0
hotel_name     0
room_code      0
room_name      0
rate_key       0
price          0
board_name     0
dtype: int64


In [42]:
# ---------------------------------------------------------
# SELECT BATCH 6
# ---------------------------------------------------------
# Recalculate the remaining destinations from the collection
# log so the process remains resumable and avoids duplicate
# API requests.
# ---------------------------------------------------------

completed_destinations = set(
    accommodation_log_df.loc[
        accommodation_log_df["status"] == "success",
        "destination"
    ]
)

remaining_destinations = [
    destination
    for destination in all_destinations
    if destination not in completed_destinations
]

# Select the next 10 destinations.
batch_size = 15
current_batch = remaining_destinations[:batch_size]

print("Processed destinations:", len(completed_destinations))
print("Remaining destinations:", len(remaining_destinations))

print("\nBatch 6 size:", len(current_batch))

for i, destination in enumerate(current_batch, start=1):
    print(f"{i}. {destination}")

Processed destinations: 35
Remaining destinations: 15

Batch 6 size: 15
1. Kolkata
2. Bengaluru
3. Hyderabad
4. Chennai
5. Pune
6. Ahmedabad
7. Bhopal
8. Indore
9. Ranchi
10. Bhubaneswar
11. Shillong
12. Kaziranga
13. Jim Corbett
14. Ranthambore
15. Pahalgam


In [43]:
# ---------------------------------------------------------
# COLLECT ACCOMMODATION DATA — FINAL BATCH
# ---------------------------------------------------------
# Collect the remaining 15 destinations.
# Progress is saved after every destination.
# ---------------------------------------------------------

for destination in current_batch:

    print(f"\n{'=' * 60}")
    print(f"Collecting accommodation: {destination}")
    print(f"{'=' * 60}")

    try:
        # -----------------------------------------------------
        # 1. Find validated destination coordinates.
        # -----------------------------------------------------
        location_match = destination_locations[
            destination_locations["destination"].str.lower()
            == destination.lower()
        ]

        if location_match.empty:
            raise ValueError("Destination coordinates not found.")

        location = location_match.iloc[0]

        latitude = float(location["latitude"])
        longitude = float(location["longitude"])

        # -----------------------------------------------------
        # 2. Generate Hotelbeds authentication signature.
        # -----------------------------------------------------
        timestamp = str(int(time.time()))

        signature_string = (
            HOTELBEDS_API_KEY
            + HOTELBEDS_SECRET
            + timestamp
        )

        signature = hashlib.sha256(
            signature_string.encode("utf-8")
        ).hexdigest()

        # -----------------------------------------------------
        # 3. Prepare availability request.
        # -----------------------------------------------------
        request_body = {
            "stay": {
                "checkIn": check_in,
                "checkOut": check_out
            },
            "occupancies": [
                {
                    "rooms": 1,
                    "adults": 1,
                    "children": 0
                }
            ],
            "geolocation": {
                "latitude": latitude,
                "longitude": longitude,
                "radius": 20,
                "unit": "km"
            }
        }

        request_headers = {
            "Accept": "application/json",
            "Content-Type": "application/json",
            "Api-key": HOTELBEDS_API_KEY,
            "X-Signature": signature
        }

        # -----------------------------------------------------
        # 4. Send Hotelbeds request.
        # -----------------------------------------------------
        response = requests.post(
            availability_url,
            headers=request_headers,
            json=request_body,
            timeout=60
        )

        response.raise_for_status()

        # -----------------------------------------------------
        # 5. Parse the API response.
        # -----------------------------------------------------
        destination_accommodation = parse_hotelbeds_response(
            response.json(),
            destination=destination,
            check_in=check_in,
            check_out=check_out
        )

        records_count = len(destination_accommodation)

        # -----------------------------------------------------
        # 6. Append accommodation records.
        # -----------------------------------------------------
        if records_count > 0:

            accommodation_raw_df = pd.concat(
                [
                    accommodation_raw_df,
                    destination_accommodation
                ],
                ignore_index=True
            )

        # -----------------------------------------------------
        # 7. Determine availability.
        # -----------------------------------------------------
        if records_count > 0:
            availability_status = "AVAILABLE"
        else:
            availability_status = "NO_AVAILABILITY"

        # -----------------------------------------------------
        # 8. Update collection log.
        # -----------------------------------------------------
        new_log = pd.DataFrame([
            {
                "destination": destination,
                "status": "success",
                "records_returned": records_count,
                "error": None,
                "availability_status": availability_status
            }
        ])

        accommodation_log_df = pd.concat(
            [
                accommodation_log_df,
                new_log
            ],
            ignore_index=True
        )

        # -----------------------------------------------------
        # 9. Save immediately.
        # -----------------------------------------------------
        accommodation_raw_df.to_csv(
            accommodation_data_path,
            index=False
        )

        accommodation_log_df.to_csv(
            collection_log_path,
            index=False
        )

        print(
            f"Success → {records_count} records "
            f"({availability_status})"
        )

        # Small delay between requests.
        time.sleep(1)

    except Exception as e:

        # -----------------------------------------------------
        # 10. Record API failures without stopping the batch.
        # -----------------------------------------------------
        error_message = str(e)

        new_log = pd.DataFrame([
            {
                "destination": destination,
                "status": "failed",
                "records_returned": 0,
                "error": error_message,
                "availability_status": "API_FAILED"
            }
        ])

        accommodation_log_df = pd.concat(
            [
                accommodation_log_df,
                new_log
            ],
            ignore_index=True
        )

        accommodation_log_df.to_csv(
            collection_log_path,
            index=False
        )

        print(f"FAILED → {destination}")
        print("Reason:", error_message)

print("\nFinal accommodation collection completed.")


Success → 237 records (AVAILABLE)

Success → 408 records (AVAILABLE)

Success → 132 records (AVAILABLE)

Success → 149 records (AVAILABLE)

Success → 306 records (AVAILABLE)

Success → 204 records (AVAILABLE)

Success → 48 records (AVAILABLE)

Success → 100 records (AVAILABLE)

Success → 0 records (NO_AVAILABILITY)

Success → 102 records (AVAILABLE)

Success → 0 records (NO_AVAILABILITY)

Success → 0 records (NO_AVAILABILITY)

Success → 0 records (NO_AVAILABILITY)

Success → 3 records (AVAILABLE)

FAILED → Pahalgam
Reason: 403 Client Error: Forbidden for url: https://api.test.hotelbeds.com/hotel-api/1.0/hotels

Final accommodation collection completed.


In [45]:
# ---------------------------------------------------------
# FINAL COLLECTION STATUS CHECK
# ---------------------------------------------------------
# Separate successfully processed destinations from the
# destination that experienced an actual API failure.
# ---------------------------------------------------------

print("Total master destinations:", len(all_destinations))

successful_destinations = set(
    accommodation_log_df.loc[
        accommodation_log_df["status"] == "success",
        "destination"
    ]
)

failed_destinations = accommodation_log_df[
    accommodation_log_df["status"] == "failed"
]["destination"].unique().tolist()

print(
    "Successfully processed:",
    len(successful_destinations)
)

print(
    "API failures:",
    len(failed_destinations)
)

print("\nFailed destinations:")
print(failed_destinations)

print("\nMissing from successful collection:")

missing_destinations = [
    destination
    for destination in all_destinations
    if destination not in successful_destinations
]

print(missing_destinations)

Total master destinations: 50
Successfully processed: 49
API failures: 1

Failed destinations:
['Pahalgam']

Missing from successful collection:
['Pahalgam']


In [46]:
# ---------------------------------------------------------
# INSPECT PAHALGAM FAILURE
# ---------------------------------------------------------
# Retrieve the exact failure information from the collection
# log before attempting another API request.
# ---------------------------------------------------------

pahalgam_log = accommodation_log_df[
    accommodation_log_df["destination"] == "Pahalgam"
]

display(
    pahalgam_log[
        [
            "destination",
            "status",
            "records_returned",
            "error",
            "availability_status"
        ]
    ]
)

,destination,status,records_returned,error,availability_status
49,Pahalgam,failed,0,403 Client Error: Forbidden for url: https://a...,API_FAILED


In [47]:
# ---------------------------------------------------------
# VERIFY PAHALGAM COORDINATES
# ---------------------------------------------------------
# Make sure the coordinates being sent to Hotelbeds are the
# same validated coordinates used during our destination
# and weather collection.
# ---------------------------------------------------------

pahalgam_location = destination_locations[
    destination_locations["destination"].str.lower()
    == "pahalgam"
]

display(
    pahalgam_location[
        [
            "destination",
            "latitude",
            "longitude"
        ]
    ]
)

,destination,latitude,longitude
49,Pahalgam,34.032205,75.322648


In [48]:
# ---------------------------------------------------------
# RETRY PAHALGAM — ONE CONTROLLED RETRY
# ---------------------------------------------------------
# Pahalgam has correct coordinates but its previous request
# returned HTTP 403.
#
# We will make ONE retry using a fresh Hotelbeds signature.
# If it fails again, we will keep it documented as an API
# failure instead of repeatedly consuming API quota.
# ---------------------------------------------------------

destination = "Pahalgam"

try:
    # -----------------------------------------------------
    # 1. Get the validated coordinates.
    # -----------------------------------------------------
    location = destination_locations[
        destination_locations["destination"].str.lower()
        == destination.lower()
    ].iloc[0]

    latitude = float(location["latitude"])
    longitude = float(location["longitude"])

    print("Destination:", destination)
    print("Latitude:", latitude)
    print("Longitude:", longitude)

    # -----------------------------------------------------
    # 2. Generate a fresh authentication signature.
    # -----------------------------------------------------
    timestamp = str(int(time.time()))

    signature_string = (
        HOTELBEDS_API_KEY
        + HOTELBEDS_SECRET
        + timestamp
    )

    signature = hashlib.sha256(
        signature_string.encode("utf-8")
    ).hexdigest()

    # -----------------------------------------------------
    # 3. Prepare the same validated availability request.
    # -----------------------------------------------------
    request_body = {
        "stay": {
            "checkIn": check_in,
            "checkOut": check_out
        },
        "occupancies": [
            {
                "rooms": 1,
                "adults": 1,
                "children": 0
            }
        ],
        "geolocation": {
            "latitude": latitude,
            "longitude": longitude,
            "radius": 20,
            "unit": "km"
        }
    }

    request_headers = {
        "Accept": "application/json",
        "Content-Type": "application/json",
        "Api-key": HOTELBEDS_API_KEY,
        "X-Signature": signature
    }

    # -----------------------------------------------------
    # 4. Send ONE retry request.
    # -----------------------------------------------------
    response = requests.post(
        availability_url,
        headers=request_headers,
        json=request_body,
        timeout=60
    )

    print("Status code:", response.status_code)

    response.raise_for_status()

    # -----------------------------------------------------
    # 5. Parse the successful response.
    # -----------------------------------------------------
    destination_accommodation = parse_hotelbeds_response(
        response.json(),
        destination=destination,
        check_in=check_in,
        check_out=check_out
    )

    records_count = len(destination_accommodation)

    # -----------------------------------------------------
    # 6. Add any returned accommodation records.
    # -----------------------------------------------------
    if records_count > 0:

        accommodation_raw_df = pd.concat(
            [
                accommodation_raw_df,
                destination_accommodation
            ],
            ignore_index=True
        )

        availability_status = "AVAILABLE"

    else:

        availability_status = "NO_AVAILABILITY"

    # -----------------------------------------------------
    # 7. Replace the previous failed Pahalgam log entry
    #    with the successful retry result.
    # -----------------------------------------------------
    accommodation_log_df = accommodation_log_df[
        accommodation_log_df["destination"] != destination
    ]

    new_log = pd.DataFrame([
        {
            "destination": destination,
            "status": "success",
            "records_returned": records_count,
            "error": None,
            "availability_status": availability_status
        }
    ])

    accommodation_log_df = pd.concat(
        [
            accommodation_log_df,
            new_log
        ],
        ignore_index=True
    )

    # -----------------------------------------------------
    # 8. Save both datasets.
    # -----------------------------------------------------
    accommodation_raw_df.to_csv(
        accommodation_data_path,
        index=False
    )

    accommodation_log_df.to_csv(
        collection_log_path,
        index=False
    )

    print(
        f"Success → {records_count} records "
        f"({availability_status})"
    )

except Exception as e:

    # Keep Pahalgam as an API failure if the retry fails.
    print("Pahalgam retry failed.")
    print("Reason:", str(e))

Destination: Pahalgam
Latitude: 34.0322048
Longitude: 75.3226479
Status code: 403
Pahalgam retry failed.
Reason: 403 Client Error: Forbidden for url: https://api.test.hotelbeds.com/hotel-api/1.0/hotels


In [49]:
# ---------------------------------------------------------
# DIAGNOSTIC TEST — GOA
# ---------------------------------------------------------
# Goa previously returned accommodation successfully.
#
# If Goa now returns 403:
#     The API quota/request limit is the likely problem.
#
# If Goa returns 200:
#     The API is still working, so Pahalgam's 403 is likely
#     destination-specific or related to its inventory/request.
# ---------------------------------------------------------

destination = "Goa"

# Get Goa's validated coordinates.
location = destination_locations[
    destination_locations["destination"].str.lower()
    == destination.lower()
].iloc[0]

latitude = float(location["latitude"])
longitude = float(location["longitude"])

# Create a fresh Hotelbeds signature.
timestamp = str(int(time.time()))

signature_string = (
    HOTELBEDS_API_KEY
    + HOTELBEDS_SECRET
    + timestamp
)

signature = hashlib.sha256(
    signature_string.encode("utf-8")
).hexdigest()

# Use exactly the same request structure as before.
request_body = {
    "stay": {
        "checkIn": check_in,
        "checkOut": check_out
    },
    "occupancies": [
        {
            "rooms": 1,
            "adults": 1,
            "children": 0
        }
    ],
    "geolocation": {
        "latitude": latitude,
        "longitude": longitude,
        "radius": 20,
        "unit": "km"
    }
}

request_headers = {
    "Accept": "application/json",
    "Content-Type": "application/json",
    "Api-key": HOTELBEDS_API_KEY,
    "X-Signature": signature
}

# Send ONE diagnostic request.
response = requests.post(
    availability_url,
    headers=request_headers,
    json=request_body,
    timeout=60
)

print("Destination:", destination)
print("Status code:", response.status_code)
print("Response preview:")
print(response.text[:500])

Destination: Goa
Status code: 403
Response preview:
{
    "error": "Quota exceeded"
}


In [50]:
# ---------------------------------------------------------
# FINAL ACCOMMODATION DATASET CHECK
# ---------------------------------------------------------
# We are temporarily accepting 49/50 destinations because
# the Hotelbeds API quota has been exhausted.
#
# Pahalgam remains explicitly marked as API_FAILED.
# ---------------------------------------------------------

print("Total accommodation records:", len(accommodation_raw_df))

print(
    "Unique destinations with accommodation:",
    accommodation_raw_df["destination"].nunique()
)

print("\nCollection status:")
print(
    accommodation_log_df["status"].value_counts()
)

print("\nAvailability status:")
print(
    accommodation_log_df["availability_status"].value_counts()
)

# ---------------------------------------------------------
# Check which master destinations are missing from the
# actual accommodation records.
# ---------------------------------------------------------

accommodation_destinations = set(
    accommodation_raw_df["destination"].unique()
)

missing_accommodation = [
    destination
    for destination in all_destinations
    if destination not in accommodation_destinations
]

print("\nDestinations without accommodation records:")
print(missing_accommodation)

# ---------------------------------------------------------
# Check important accommodation fields.
# ---------------------------------------------------------

important_columns = [
    "destination",
    "hotel_code",
    "hotel_name",
    "room_code",
    "room_name",
    "rate_key",
    "price",
    "board_name"
]

print("\nMissing important values:")

print(
    accommodation_raw_df[
        important_columns
    ].isna().sum()
)

Total accommodation records: 4835
Unique destinations with accommodation: 39

Collection status:
status
success    49
failed      1
Name: count, dtype: int64

Availability status:
availability_status
AVAILABLE          39
NO_AVAILABILITY    10
API_FAILED          1
Name: count, dtype: int64

Destinations without accommodation records:
['Kodaikanal', 'Coorg', 'Wayanad', 'Mahabalipuram', 'Andaman', 'Ladakh', 'Ranchi', 'Shillong', 'Kaziranga', 'Jim Corbett', 'Pahalgam']

Missing important values:
destination    0
hotel_code     0
hotel_name     0
room_code      0
room_name      0
rate_key       0
price          0
board_name     0
dtype: int64


In [51]:
# ---------------------------------------------------------
# SAVE FINAL ACCOMMODATION CHECKPOINT
# ---------------------------------------------------------
# Save the collected accommodation records and collection
# status separately.
#
# We keep the raw accommodation data untouched because it
# contains the hotel → room → rate information needed later
# for price and recommendation logic.
# ---------------------------------------------------------

# Save the complete accommodation dataset collected so far.
accommodation_raw_df.to_csv(
    accommodation_data_path,
    index=False
)

# Save the complete collection log.
accommodation_log_df.to_csv(
    collection_log_path,
    index=False
)

print("Accommodation dataset saved.")
print("Collection log saved.")

print("\nAccommodation records:", len(accommodation_raw_df))
print(
    "Destinations with accommodation:",
    accommodation_raw_df["destination"].nunique()
)

print("\nCollection status:")
print(
    accommodation_log_df["status"].value_counts()
)

print("\nAvailability status:")
print(
    accommodation_log_df["availability_status"].value_counts()
)

Accommodation dataset saved.
Collection log saved.

Accommodation records: 4835
Destinations with accommodation: 39

Collection status:
status
success    49
failed      1
Name: count, dtype: int64

Availability status:
availability_status
AVAILABLE          39
NO_AVAILABILITY    10
API_FAILED          1
Name: count, dtype: int64


In [5]:
# ---------------------------------------------------------
# PAHALGAM — RETRY AFTER HOTELBEDS QUOTA RESET
# ---------------------------------------------------------
# IMPORTANT:
# Run this cell ONLY after the Hotelbeds API quota has reset.
#
# What this cell does:
# 1. Checks Pahalgam's validated coordinates.
# 2. Creates a fresh Hotelbeds authentication signature.
# 3. Requests accommodation for Pahalgam.
# 4. If successful, appends the records to the existing
#    accommodation_raw_df.
# 5. Removes the old failed Pahalgam log entry.
# 6. Adds the new successful result to accommodation_log_df.
# 7. Saves both datasets.
#
# It does NOT modify accommodation records for any other
# destination.
# ---------------------------------------------------------

destination_locations = pd.read_csv(
    "../data/raw/places/destination_locations.csv"
)

destination = "Pahalgam"

print("=" * 60)
print("Retrying accommodation collection:", destination)
print("=" * 60)

try:

    # -----------------------------------------------------
    # 1. Get Pahalgam's validated coordinates.
    # -----------------------------------------------------
    location_match = destination_locations[
        destination_locations["destination"].str.lower()
        == destination.lower()
    ]

    if location_match.empty:
        raise ValueError(
            "Pahalgam coordinates were not found."
        )

    location = location_match.iloc[0]

    latitude = float(location["latitude"])
    longitude = float(location["longitude"])

    print("Latitude:", latitude)
    print("Longitude:", longitude)

    # -----------------------------------------------------
    # 2. Generate a fresh Hotelbeds signature.
    # -----------------------------------------------------
    timestamp = str(int(time.time()))

    signature_string = (
        HOTELBEDS_API_KEY
        + HOTELBEDS_SECRET
        + timestamp
    )

    signature = hashlib.sha256(
        signature_string.encode("utf-8")
    ).hexdigest()

    # -----------------------------------------------------
    # 3. Prepare the Hotelbeds availability request.
    # -----------------------------------------------------
    request_body = {
        "stay": {
            "checkIn": check_in,
            "checkOut": check_out
        },
        "occupancies": [
            {
                "rooms": 1,
                "adults": 1,
                "children": 0
            }
        ],
        "geolocation": {
            "latitude": latitude,
            "longitude": longitude,
            "radius": 20,
            "unit": "km"
        }
    }

    request_headers = {
        "Accept": "application/json",
        "Content-Type": "application/json",
        "Api-key": HOTELBEDS_API_KEY,
        "X-Signature": signature
    }

    # -----------------------------------------------------
    # 4. Send the request.
    # -----------------------------------------------------
    response = requests.post(
        availability_url,
        headers=request_headers,
        json=request_body,
        timeout=60
    )

    print("Status code:", response.status_code)

    response.raise_for_status()

    # -----------------------------------------------------
    # 5. Parse the successful Hotelbeds response.
    # -----------------------------------------------------
    pahalgam_accommodation = parse_hotelbeds_response(
        response.json(),
        destination=destination,
        check_in=check_in,
        check_out=check_out
    )

    records_count = len(pahalgam_accommodation)

    print(
        f"Pahalgam records returned: {records_count}"
    )

    # -----------------------------------------------------
    # 6. Remove any OLD Pahalgam records.
    #
    # Normally there are none because the previous attempt
    # failed, but this prevents duplicate records if this
    # retry cell is accidentally run after a successful retry.
    # -----------------------------------------------------
    accommodation_raw_df = accommodation_raw_df[
        accommodation_raw_df["destination"] != destination
    ].copy()

    # -----------------------------------------------------
    # 7. Add the newly collected Pahalgam records.
    # -----------------------------------------------------
    if records_count > 0:

        accommodation_raw_df = pd.concat(
            [
                accommodation_raw_df,
                pahalgam_accommodation
            ],
            ignore_index=True
        )

        availability_status = "AVAILABLE"

    else:

        # API succeeded but returned no matching accommodation.
        availability_status = "NO_AVAILABILITY"

    # -----------------------------------------------------
    # 8. Remove the old failed Pahalgam entry from the log.
    # -----------------------------------------------------
    accommodation_log_df = accommodation_log_df[
        accommodation_log_df["destination"] != destination
    ].copy()

    # -----------------------------------------------------
    # 9. Add the new Pahalgam result.
    # -----------------------------------------------------
    new_log = pd.DataFrame([
        {
            "destination": destination,
            "status": "success",
            "records_returned": records_count,
            "error": None,
            "availability_status": availability_status
        }
    ])

    accommodation_log_df = pd.concat(
        [
            accommodation_log_df,
            new_log
        ],
        ignore_index=True
    )

    # -----------------------------------------------------
    # 10. Save the updated accommodation dataset.
    # -----------------------------------------------------
    accommodation_raw_df.to_csv(
        accommodation_data_path,
        index=False
    )

    # -----------------------------------------------------
    # 11. Save the updated collection log.
    # -----------------------------------------------------
    accommodation_log_df.to_csv(
        collection_log_path,
        index=False
    )

    print("\nPahalgam update completed successfully.")
    print(
        f"Status: {availability_status}"
    )

    print(
        f"Records added: {records_count}"
    )

    print(
        f"\nTotal accommodation records now: "
        f"{len(accommodation_raw_df)}"
    )

    print(
        "Unique destinations with accommodation:",
        accommodation_raw_df["destination"].nunique()
    )

except Exception as e:

    # -----------------------------------------------------
    # If the quota has NOT reset, keep the existing data
    # untouched and report the error.
    # -----------------------------------------------------

    print("\nPahalgam retry failed.")
    print("Reason:", str(e))

    print(
        "\nExisting accommodation dataset was NOT modified."
    )

Retrying accommodation collection: Pahalgam
Latitude: 34.0322048
Longitude: 75.3226479

Pahalgam retry failed.
Reason: name 'time' is not defined

Existing accommodation dataset was NOT modified.


In [1]:
import pandas as pd

# Load the accommodation records already collected
accommodation_df = pd.read_csv(
    "../data/raw/accommodation/accommodation_raw.csv"
)

print("Rows:", len(accommodation_df))
print("Columns:", accommodation_df.columns.tolist())

Rows: 4835
Columns: ['destination', 'check_in', 'check_out', 'hotel_code', 'hotel_name', 'hotel_category', 'hotel_category_name', 'destination_code', 'destination_name', 'zone_code', 'zone_name', 'hotel_latitude', 'hotel_longitude', 'room_code', 'room_name', 'rate_key', 'rate_class', 'rate_type', 'price', 'allotment', 'payment_type', 'board_code', 'board_name', 'rooms', 'adults', 'children', 'offer_name', 'offer_amount', 'cancellation_amount', 'cancellation_from']


In [2]:
# Basic validation of the accommodation dataset

print("Rows:", len(accommodation_df))
print("Unique destinations:", accommodation_df["destination"].nunique())

print("\nMissing important values:")
print(
    accommodation_df[
        [
            "destination",
            "hotel_code",
            "hotel_name",
            "room_code",
            "room_name",
            "rate_key",
            "price",
            "board_name"
        ]
    ].isna().sum()
)

print("\nDuplicate rows:", accommodation_df.duplicated().sum())

Rows: 4835
Unique destinations: 39

Missing important values:
destination    0
hotel_code     0
hotel_name     0
room_code      0
room_name      0
rate_key       0
price          0
board_name     0
dtype: int64

Duplicate rows: 310


In [3]:
# Inspect a few duplicate accommodation records

duplicates = accommodation_df[
    accommodation_df.duplicated(keep=False)
].sort_values(
    by=["destination", "hotel_code"]
)

print("Duplicate rows:", len(duplicates))

display(duplicates.head(20))

Duplicate rows: 620


,destination,check_in,check_out,hotel_code,hotel_name,hotel_category,hotel_category_name,destination_code,destination_name,zone_code,...,payment_type,board_code,board_name,rooms,adults,children,offer_name,offer_amount,cancellation_amount,cancellation_from
3087,Amritsar,2026-09-15,2026-09-16,691291,"The Fern Residency, Amritsar",3EST,3 STARS,ATQ,Amritsar,1,...,AT_WEB,BB,BED AND BREAKFAST,1,1,0,NaN,NaN,53.71,2026-09-11T23:59:00+05:30
3088,Amritsar,2026-09-15,2026-09-16,691291,"The Fern Residency, Amritsar",3EST,3 STARS,ATQ,Amritsar,1,...,AT_WEB,BB,BED AND BREAKFAST,1,1,0,NaN,NaN,53.71,2026-09-11T23:59:00+05:30
3093,Amritsar,2026-09-15,2026-09-16,691291,"The Fern Residency, Amritsar",3EST,3 STARS,ATQ,Amritsar,1,...,AT_WEB,BB,BED AND BREAKFAST,1,1,0,NaN,NaN,59.20,2026-09-11T23:59:00+05:30
3094,Amritsar,2026-09-15,2026-09-16,691291,"The Fern Residency, Amritsar",3EST,3 STARS,ATQ,Amritsar,1,...,AT_WEB,BB,BED AND BREAKFAST,1,1,0,NaN,NaN,59.20,2026-09-11T23:59:00+05:30
3099,Amritsar,2026-09-15,2026-09-16,691291,"The Fern Residency, Amritsar",3EST,3 STARS,ATQ,Amritsar,1,...,AT_WEB,BB,BED AND BREAKFAST,1,1,0,NaN,NaN,59.20,2026-09-11T23:59:00+05:30
3100,Amritsar,2026-09-15,2026-09-16,691291,"The Fern Residency, Amritsar",3EST,3 STARS,ATQ,Amritsar,1,...,AT_WEB,BB,BED AND BREAKFAST,1,1,0,NaN,NaN,59.20,2026-09-11T23:59:00+05:30
3051,Amritsar,2026-09-15,2026-09-16,1027154,Hotel Dg By Divudecom,3EST,3 STARS,ATQ,Amritsar,1,...,AT_WEB,RO,ROOM ONLY,1,1,0,NaN,NaN,8009.07,2026-09-13T23:59:00+05:30
3052,Amritsar,2026-09-15,2026-09-16,1027154,Hotel Dg By Divudecom,3EST,3 STARS,ATQ,Amritsar,1,...,AT_WEB,RO,ROOM ONLY,1,1,0,NaN,NaN,8009.07,2026-09-13T23:59:00+05:30
3054,Amritsar,2026-09-15,2026-09-16,1027154,Hotel Dg By Divudecom,3EST,3 STARS,ATQ,Amritsar,1,...,AT_WEB,RO,ROOM ONLY,1,1,0,NaN,NaN,8009.07,2026-09-13T23:59:00+05:30
3055,Amritsar,2026-09-15,2026-09-16,1027154,Hotel Dg By Divudecom,3EST,3 STARS,ATQ,Amritsar,1,...,AT_WEB,RO,ROOM ONLY,1,1,0,NaN,NaN,8009.07,2026-09-13T23:59:00+05:30


In [4]:
# Save the collected accommodation data

output_path = "../data/cleaned/accommodation_data.csv"

accommodation_df.to_csv(
    output_path,
    index=False
)

print("Saved accommodation data successfully.")
print("Rows saved:", len(accommodation_df))
print("Path:", output_path)

Saved accommodation data successfully.
Rows saved: 4835
Path: ../data/cleaned/accommodation_data.csv
